# Pauli Propagation: Basic Usage

This notebook demonstrates two usage patterns for the Pauli Propagation Executor:
1. **Native API** (strict type enforcement): Using `PauliPropagationCircuit` and `PauliPropagationObservable` directly
2. **Generic API** (transparent conversion): Using generic `QuantumCircuit` and `QuantumOperator` which are automatically transpiled to native types

## Variant 1: Native Types (Strict API)

In [ ]:
import numpy as np

from executor import Executor, Parameters
from executor.pauli_propagation import (
    PauliPropagationCircuit,
    PauliPropagationObservable,
)
from executor.pauli_propagation.symmetry import PermutationSymmetry

In [ ]:
theta = Parameters('theta', 1)
circuit = PauliPropagationCircuit(2)
circuit.h(0)
circuit.cx(0, 1)
circuit.ry(1, theta[0])

observable = PauliPropagationObservable(
    ['ZI', 'IZ'],
    [0.5, 0.5],
    symmetry_strategy=PermutationSymmetry(),
)

executor = Executor.create(
    'pauli_propagation',
    shots=2000,
    seed=7,
    truncate_threshold=1e-10,
    symmetry_strategy=PermutationSymmetry(),
)

print('=== Native API - Circuit and Observable ===')
print(f'Circuit type: {type(circuit).__name__}')
print(f'Observable type: {type(observable).__name__}')
print(f'Observable symmetry: {observable.symmetry.name}')

In [ ]:
exp_0 = executor.expectation_value(circuit, observable, **{'theta[0]': 0.0})
exp_pi = executor.expectation_value(circuit, observable, **{'theta[0]': np.pi})

print('\n=== Native API Results ===')
print(f'<O>(theta=0): {exp_0}')
print(f'<O>(theta=pi): {exp_pi}')

## Variant 2: Generic Types (Transparent Conversion)

In [ ]:
from executor import QuantumCircuit, QuantumOperator

generic_circuit = QuantumCircuit(2)
qiskit_circuit = generic_circuit._qiskit_circuit
qiskit_circuit.h(0)
qiskit_circuit.cx(0, 1)
qiskit_circuit.ry(1, theta[0])

generic_observable = QuantumOperator(
    paulis=['ZI', 'IZ'],
    coeffs=[0.5, 0.5],
)

print('\n=== Generic API Types (auto-transpiled) ===')
print(f'Generic circuit type: {type(generic_circuit).__name__}')
print(f'Generic observable type: {type(generic_observable).__name__}')

In [ ]:
exp_0_generic = executor.expectation_value(generic_circuit, generic_observable, **{'theta[0]': 0.0})
exp_pi_generic = executor.expectation_value(generic_circuit, generic_observable, **{'theta[0]': np.pi})

print('\n=== Generic API Results ===')
print(f'<O>(theta=0): {exp_0_generic}')
print(f'<O>(theta=pi): {exp_pi_generic}')

print('\n=== Verification: Native vs Generic ===')
print(f'Results match (theta=0): {np.isclose(exp_0, exp_0_generic)}')
print(f'Results match (theta=pi): {np.isclose(exp_pi, exp_pi_generic)}')